## 03 — Pitch Distribution EDA

Visual exploration of how pitch selection varies across game contexts using June 2024 Statcast data enriched with 2023 pitcher/batter arsenal stats.

**Sections:**
1. Target Distribution — overall pitch type frequency and pitcher handedness split
2. Count-Situation Analysis — how balls and strikes shift pitch selection
3. Handedness Matchup — pitch mix across pitcher × batter handedness combinations
4. Game Context Features — pitch number, score differential, inning (`n_thruorder_pitcher` exploratory only)
5. Prior-Year Arsenal Usage — how well `pitch_usage_*` columns predict actual June 2024 usage

See **02_data_quality** for the column-level audit. See **04_feature_importance_ml** for feature selection experiments.

In [ ]:
%matplotlib inline

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import pybaseball
from pybaseball import cache, statcast

sys.path.append(str(Path("..").resolve()))
from utils.features.enrichment import enrich_statcast
from utils.features.feature_names import (
    BATTER_OUTCOME_COLUMNS,
    BATTER_USAGE_COLUMNS,
    CANDIDATE_GAME_STATE_COLUMNS,
    EDA_COLUMNS,
    LABEL_COLUMN,
    PITCHER_OUTCOME_COLUMNS,
    PITCHER_USAGE_COLUMNS,
    PITCH_TYPES,
)

DATA_DIR = Path("..") / "data" / "cache"
DATA_DIR.mkdir(parents=True, exist_ok=True)
cache.config.cache_directory = str(DATA_DIR.resolve())
cache.config.save()
cache.enable()

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print("pybaseball :", pybaseball.__version__)
print("pandas     :", pd.__version__)
print("cache dir  :", cache.config.cache_directory)

In [ ]:
SEASON = 2024
PRIOR  = SEASON - 1

df = statcast("2024-06-01", "2024-06-30")
print(f"Raw shape: {df.shape}")

df_enr = enrich_statcast(df, PRIOR)
print(f"Enriched shape: {df_enr.shape}")
if "pitch_usage_FF" in df_enr.columns:
    print(f"Pitcher usage coverage (pitch_usage_FF non-null): {df_enr['pitch_usage_FF'].notna().mean():.1%}")

In [ ]:
available    = [c for c in EDA_COLUMNS if c in df_enr.columns]
missing_cols = [c for c in EDA_COLUMNS if c not in df_enr.columns]

model_df = df_enr[available + [LABEL_COLUMN]].copy()
model_df = model_df[model_df[LABEL_COLUMN].isin(PITCH_TYPES)].reset_index(drop=True)

print(f"model_df shape       : {model_df.shape}")
print(f"EDA_COLUMNS total    : {len(EDA_COLUMNS)}")
print(f"Found in enriched frame: {len(available)}")
if missing_cols:
    preview = missing_cols[:5]
    ellipsis = "..." if len(missing_cols) > 5 else ""
    print(f"Missing ({len(missing_cols)})           : {preview}{ellipsis}")
print(f"\nPitch type distribution:\n{model_df[LABEL_COLUMN].value_counts()}")

---
## Section 1 — Target Distribution

Understand class imbalance and how pitch mix varies by pitcher handedness before any feature work.

In [ ]:
### Visual 1 — Overall pitch type frequency
counts = model_df[LABEL_COLUMN].value_counts()
total  = counts.sum()

fig, ax = plt.subplots(figsize=(8, 5))
palette = sns.color_palette("tab10", len(counts))
bars = ax.barh(counts.index, counts.values, color=palette)

for bar, cnt in zip(bars, counts.values):
    ax.text(
        bar.get_width() + total * 0.004,
        bar.get_y() + bar.get_height() / 2,
        f"{cnt:,}  ({cnt / total:.1%})",
        va="center", fontsize=9,
    )

ax.set_xlabel("Pitch count")
ax.set_title("Pitch Type Distribution — June 2024", fontsize=13)
ax.set_xlim(0, counts.max() * 1.30)
ax.invert_yaxis()
sns.despine(left=True, bottom=False)
plt.tight_layout()
plt.show()

In [ ]:
### Visual 2 — Pitch mix by pitcher handedness (normalised within L / R)
hand_mix = (
    model_df.groupby(["p_throws", LABEL_COLUMN])
    .size()
    .unstack(fill_value=0)
    .apply(lambda row: row / row.sum(), axis=1)
)
### Reorder columns by overall frequency so bars are visually consistent
hand_mix = hand_mix.reindex(columns=counts.index, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
hand_mix.T.plot(kind="bar", ax=ax, color=["#4c72b0", "#dd8452"], width=0.65, edgecolor="white")
ax.set_xlabel("Pitch Type")
ax.set_ylabel("Proportion of pitches thrown")
ax.set_title("Pitch Mix by Pitcher Handedness (normalised within L / R)", fontsize=13)
ax.legend(title="p_throws", loc="upper right")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
plt.xticks(rotation=0)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
### Visual 3 — Data prep: pitch usage normalised within each (p_throws × stand) matchup
handedness_mix = (
    model_df.groupby(["p_throws", "stand", LABEL_COLUMN])
    .size()
    .unstack(fill_value=0)
    .apply(lambda row: row / row.sum(), axis=1)   # normalise within matchup
)
handedness_mix = handedness_mix.reindex(columns=counts.index, fill_value=0)
handedness_mix = handedness_mix.reset_index()

### Sanity check
print(handedness_mix[["p_throws", "stand"]].value_counts().sort_index())
print(handedness_mix.head())

In [ ]:
import math

pitch_types_plot = [pt for pt in counts.index if pt in handedness_mix.columns]
overall_rate = counts / counts.sum()   # used for delta layer

### Figure layout
ncols = 5
nrows = math.ceil(len(pitch_types_plot) / ncols)

### Primary heatmap — absolute normalised usage %
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.8, nrows * 2.6))
axes = axes.flatten()

for ax, pt in zip(axes, pitch_types_plot):
    matrix = handedness_mix.pivot(index="p_throws", columns="stand", values=pt)
    matrix = matrix.reindex(index=["L", "R"], columns=["L", "R"]) * 100
    sns.heatmap(
        matrix, ax=ax,
        annot=True, fmt=".1f", cmap="YlOrRd",
        vmin=0, linewidths=0.5,
        cbar_kws={"label": "%", "shrink": 0.8},
    )
    ax.set_title(pt, fontweight="bold", fontsize=11)
    ax.set_xlabel("Batter (stand)", fontsize=8)
    ax.set_ylabel("Pitcher (p_throws)", fontsize=8)
    ax.tick_params(labelsize=8)

### Hide any unused subplot slots
for ax in axes[len(pitch_types_plot):]:
    ax.set_visible(False)

fig.suptitle(
    "Pitch Usage % by Pitcher × Batter Handedness\n"
    "(normalised within each matchup)",
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.show()

### Delta heatmap — usage relative to each pitch type's overall baseline
fig2, axes2 = plt.subplots(nrows, ncols, figsize=(ncols * 2.8, nrows * 2.6))
axes2 = axes2.flatten()

for ax, pt in zip(axes2, pitch_types_plot):
    matrix = handedness_mix.pivot(index="p_throws", columns="stand", values=pt)
    matrix = matrix.reindex(index=["L", "R"], columns=["L", "R"]) * 100
    delta = matrix - (overall_rate[pt] * 100)
    abs_max = max(abs(delta.values.min()), abs(delta.values.max()), 0.5)
    sns.heatmap(
        delta, ax=ax,
        annot=True, fmt="+.1f", cmap="RdBu_r",
        center=0, vmin=-abs_max, vmax=abs_max,
        linewidths=0.5,
        cbar_kws={"label": "pp vs baseline", "shrink": 0.8},
    )
    ax.set_title(pt, fontweight="bold", fontsize=11)
    ax.set_xlabel("Batter (stand)", fontsize=8)
    ax.set_ylabel("Pitcher (p_throws)", fontsize=8)
    ax.tick_params(labelsize=8)

for ax in axes2[len(pitch_types_plot):]:
    ax.set_visible(False)

fig2.suptitle(
    "Pitch Usage Deviation from Overall Baseline (pp)\n"
    "Red = overused vs baseline  |  Blue = underused",
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.show()